# Один из возможных алгоритмов скрейпинга

* указать в коде адрес интересующего сайта: откуда вы хотите скачать данные?
для этого используется библиотека requests

* сохранить веб-страницу (html-код страницы)

* выбрать данные, которые нужно собрать (используется BeautifulSoup)

* записать данные в csv-файл.

Если нужно соскрейпить несколько страниц - повторяем процесс для каждой из них.

**Наша задача:** выгрузить недавние новости в датафрейм `pandas`, чтобы потом сохранить все в csv-файл.

Сначала сгрузим весь html-код страницы и сохраним его в отдельную переменную. Для этого нам понадобится библиотека `requests`. Документация: https://requests.readthedocs.io/en/latest/

Импортируем её:

In [31]:
import requests

Сохраним ссылку на главную страницу сайта в переменную `url` для удобства и выгрузим страницу. (Разумеется, это будет работать при подключении к интернету. Если соединение будет отключено, Python выдаст `NewConnectionError`).

In [32]:
url = 'https://nplus1.ru/' # сохраняем
page = requests.get(url) # загружаем страницу по ссылке

ConnectionError: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))

Если мы просто посмотрим на объект, мы ничего особенного не увидим:

In [33]:
page  # response 200 - страница загружена

<Response [200]>

**BeautifulSoup** - python-библиотека для синтаксического разбора файлов HTML/XML. В веб-разработке «суп из тегов» (tag soup) - это слово для синтаксически или структурно некорректного HTML, написанного для веб-страницы.

Документация: https://www.crummy.com/software/BeautifulSoup/bs4/doc/

Импортируем функцию `BeautifulSoup` из библиотеки `bs4` (от *beautifulsoup4*) и заберём со страницы `page` код html в виде текста. 

Сохраним в переменную 'soup' весь HTML-код страницы. HTML-код - это "дерево тегов", формирующее контент страницы.

In [4]:
!pip install bs4

In [34]:
from bs4 import BeautifulSoup 
# название - отсылка к песне про суп из Алисы в стране чудес https://aliceinwonderland.fandom.com/wiki/Turtle_Soup

In [35]:
soup = BeautifulSoup(page.text, 'html')

Если выведем `soup` на экран, мы увидим то же самое, что в режиме разработчика или в режиме происмотра исходного кода (`view-source` через *Ctrl+U* в Google Chrome).

In [36]:
soup

<!DOCTYPE html>
<html class="bg-light-gray" lang="ru" prefix="og: http://ogp.me/ns#">
<head>
<meta charset="utf-8"/>
<meta content="width=device-width, initial-scale=1" name="viewport"/>
<title>N + 1 — главное издание о науке, технике и технологиях</title>
<link as="font" crossorigin="" href="https://staticn1.nplus1.ru/fonts/AeonikPro/AeonikPro-Regular.woff2" rel="preload" type="font/woff2"/>
<link as="font" crossorigin="" href="https://staticn1.nplus1.ru/fonts/Spectral/Spectral-Regular.woff" rel="preload" type="font/woff2"/>
<link href="/front-build/immutable/css/main.5a0ca6ad500284e2732f.css" rel="stylesheet"/>
<link href="/front-build/immutable/css/app.57ddc2610829d58883cc.css" rel="stylesheet"/>
<link href="/apple-touch-icon.png" rel="apple-touch-icon" sizes="180x180"/>
<link href="/favicon-32x32.png" rel="icon" sizes="32x32" type="image/png"/>
<link href="/favicon-16x16.png" rel="icon" sizes="16x16" type="image/png"/>
<link href="/site.webmanifest" rel="manifest"/>
<link color="#f

Для просмотра выглядит не очень удобно.  «Причешем» наш `soup` – воспользуемся методом `.prettify()` в сочетании с функцией `print()`.

In [37]:
print(soup.prettify())

<!DOCTYPE html>
<html class="bg-light-gray" lang="ru" prefix="og: http://ogp.me/ns#">
 <head>
  <meta charset="utf-8"/>
  <meta content="width=device-width, initial-scale=1" name="viewport"/>
  <title>
   N + 1 — главное издание о науке, технике и технологиях
  </title>
  <link as="font" crossorigin="" href="https://staticn1.nplus1.ru/fonts/AeonikPro/AeonikPro-Regular.woff2" rel="preload" type="font/woff2"/>
  <link as="font" crossorigin="" href="https://staticn1.nplus1.ru/fonts/Spectral/Spectral-Regular.woff" rel="preload" type="font/woff2"/>
  <link href="/front-build/immutable/css/main.5a0ca6ad500284e2732f.css" rel="stylesheet"/>
  <link href="/front-build/immutable/css/app.57ddc2610829d58883cc.css" rel="stylesheet"/>
  <link href="/apple-touch-icon.png" rel="apple-touch-icon" sizes="180x180"/>
  <link href="/favicon-32x32.png" rel="icon" sizes="32x32" type="image/png"/>
  <link href="/favicon-16x16.png" rel="icon" sizes="16x16" type="image/png"/>
  <link href="/site.webmanifest" re

В такой выдаче ориентироваться гораздо удобнее (но при желании, то же можно увидеть в браузере, на большом экране).

Чтобы сгрузить все новости с главной страницы сайта, нужно собрать все ссылки на страницы с этими новостями. Ссылки в html-файле всегда заключены в тэг `<a></a>` и имеют атрибут `href`. 

**Функция `soup.find('a')`** найдет первый в дереве тег \<a>.
Если нам нужно найти не только первый элемент, а все элементы по определенному признаку, следует использовать функцию **`soup.find_all('a')`**

In [38]:
for link in soup.find_all('a'):
    print(link.get('href'))

/
/search
/subscription?utm_source=nplus&utm_medium=nybanner&utm_content=general
https://nplus1.ru/about
https://nplus1.ru/difficult
https://nplus1.ru/adv
https://nplus1.ru/blog/2022/04/01/samotek
https://nplus1.ru/search?tags=946
https://nplus1.ru/search?tags=869
https://nplus1.ru/search?tags=874
https://nplus1.ru/search?tags=880
https://nplus1.ru/search?tags=768
https://nplus1.ru/search?tags=890
https://nplus1.ru/search?tags=871
https://nplus1.ru/search?tags=876
https://nplus1.ru/search?tags=775
https://nplus1.ru/search?tags=767
https://nplus1.ru/search?tags=771
https://nplus1.ru/search?tags=772
https://nplus1.ru/search?tags=778
https://nplus1.ru/search?tags=824
https://nplus1.ru/search?tags=908
https://nplus1.ru/search?tags=812
https://t.me/nplusone
https://vk.com/nplusone
https://ok.ru/nplus1
https://twitter.com/nplusodin
https://nplus1.ru/about
https://nplus1.ru/difficult
https://nplus1.ru/adv
https://nplus1.ru/blog/2022/04/01/samotek
https://nplus1.ru/search?tags=946
https://nplu

Ссылок много. Но нам нужны только новости – ссылки, которые начинаются со слова `/news`. Добавим условие: будем выбирать только те ссылки, в которых есть `/news`. Создадим пустой список `urls` и будем добавлять в него только ссылки, которые удовлетворяют этому условию.

In [40]:
urls = []

for link in soup.find_all('a'):
#     print(link)
    if '/news' in link.get('href'):
        if 'https://' in link.get('href'):
            urls.append(link.get('href'))
urls

['https://nplus1.ru/news/2026/11/24/ushkanik',
 'https://nplus1.ru/news/2026/11/25/fda-venglustat',
 'https://nplus1.ru/news/2026/11/25/toba-super-eruption',
 'https://nplus1.ru/news/2026/11/25/archaeospirostoma-flinti',
 'https://nplus1.ru/news/2026/11/25/fda-ribitol',
 'https://nplus1.ru/news/2026/11/25/mrdaja',
 'https://nplus1.ru/news/2026/11/24/prehospital-antibiotics-sepsis',
 'https://nplus1.ru/news/2026/11/24/pickleball-traumatic-glaucoma',
 'https://nplus1.ru/news/2026/11/24/islamic-era-shipwreck',
 'https://nplus1.ru/news/2026/11/24/solo-navigation-in-meerkats',
 'https://nplus1.ru/news/2026/11/24/cyclura-collei',
 'https://nplus1.ru/news/2026/11/24/four-decades-still-ok',
 'https://nplus1.ru/news/2026/11/24/gallic-coin-hoard',
 'https://nplus1.ru/news/2026/11/24/year-with-pig-kidney',
 'https://nplus1.ru/news/2026/11/23/leucistic-tayras',
 'https://nplus1.ru/news/2026/11/23/chiglitazar-masld',
 'https://nplus1.ru/news/2026/11/23/glp-1-hnscc',
 'https://nplus1.ru/news/2026/11

Теперь наша задача сводится к следующему: изучить одну страницу с новостью, научиться из нее вытаскивать текст и всю необходимую информацию, а потом применить весь набор действий к каждой ссылке из `full_urls` в цикле. Посмотрим на новость с индексом 1, у вас может быть другая, новости обновляются.

In [41]:
url0 = urls[1]

page0 = requests.get(url0)
soup0 = BeautifulSoup(page0.text, 'html')
soup0

<!DOCTYPE html>
<html class="bg-light-gray" lang="ru" prefix="og: http://ogp.me/ns#">
<head>
<meta charset="utf-8"/>
<meta content="width=device-width, initial-scale=1" name="viewport"/>
<title>FDA одобрило первое лекарство для лечения неврологических осложнений болезни Гоше третьего типа. Оно снижает накопление глюкоцереброзида в головном мозге</title>
<link as="font" crossorigin="" href="https://staticn1.nplus1.ru/fonts/AeonikPro/AeonikPro-Regular.woff2" rel="preload" type="font/woff2"/>
<link as="font" crossorigin="" href="https://staticn1.nplus1.ru/fonts/Spectral/Spectral-Regular.woff" rel="preload" type="font/woff2"/>
<link href="/front-build/immutable/css/main.5a0ca6ad500284e2732f.css" rel="stylesheet"/>
<link href="/front-build/immutable/css/app.57ddc2610829d58883cc.css" rel="stylesheet"/>
<link href="/apple-touch-icon.png" rel="apple-touch-icon" sizes="180x180"/>
<link href="/favicon-32x32.png" rel="icon" sizes="32x32" type="image/png"/>
<link href="/favicon-16x16.png" rel="ico

В коде каждой страницы с новостью есть часть с мета-информацией: датой, именем автора и проч. Такая информация окружена тэгом `<meta></meta>`. Посмотрим:

In [42]:
soup0.find_all('meta')

[<meta charset="utf-8"/>,
 <meta content="width=device-width, initial-scale=1" name="viewport"/>,
 <meta content="#f26e40" name="msapplication-TileColor"/>,
 <meta content="#ffffff" name="theme-color"/>,
 <meta content="8c90b02c84ac3b72" name="yandex-verification"/>,
 <meta content="b419949322895fc9106e24ed01be58ac" name="pmail-verification"/>,
 <meta content="Оно снижает накопление глюкоцереброзида в головном мозге" name="description"/>,
 <meta content="N + 1 — главное издание о науке, технике и технологиях" property="og:site_name"/>,
 <meta content="FDA одобрило первое лекарство для лечения неврологических осложнений болезни Гоше третьего типа" property="og:title"/>,
 <meta content="https://minio.nplus1.ru/app-images/media-revisions/1157231/89165273-3068-47d9-bfd9-ed66e295e486/6b06b30aebb2d_cover_share.jpeg" property="og:image"/>,
 <meta content="https://nplus1.ru/news/2026/11/25/fda-venglustat" property="og:url"/>,
 <meta content="Оно снижает накопление глюкоцереброзида в головном м

Из этого списка нам нужны части с именем автора, датой, заголовком и кратким описанием. Воспользуемся поиском по атрибуту `name`. Передадим функции `find_all()` в качестве аргумента словарь с названием и значением атрибута: 

In [43]:
soup0.find_all('meta', {'name' : 'author'}) # например, автор

[<meta content="Слава Гоменюк" name="author"/>]

Теперь выберем единственный элемент полученного списка (с индексом 0):

In [44]:
soup0.find_all('meta', {'name' : 'author'})[0]

<meta content="Слава Гоменюк" name="author"/>

Нам нужно вытащить из этого объекта `content` – имя автора. Посмотрим на атрибуты:

In [45]:
soup0.find_all('meta', {'name' : 'author'})[0].attrs

{'name': 'author', 'content': 'Слава Гоменюк'}

Как получить отсюда `content`? Очень просто, ведь это словарь! А доставать из словаря значение по ключу мы умеем.

In [46]:
author = soup0.find_all('meta', {'name' : 'author'})[0].get('content')
author

'Слава Гоменюк'

Аналогичным образом извлечем дату, заголовок и описание.

In [47]:
date = soup0.find_all('meta', {'itemprop' : 'datePublished'})[0].get('content')
title = soup0.find_all('meta', {'property' : 'og:title'})[0].get('content')

In [48]:
title

'FDA одобрило первое лекарство для лечения неврологических осложнений болезни Гоше третьего типа'

Теперь осталось совсем чуть-чуть. Написать готовую функцию для всех проделанных нами действий и применить ее в цикле для всех ссылок в списке `full_urls`. Напишем! Аргументом функции будет ссылка на новость, а возвращать она будет текст новости и всю необходимую информацию (дата, автор, сложность и проч.). Скопируем все строки кода выше.

In [49]:
def GetNews(url0):
    """
    Возвращает кортеж с url0, date, author, title.
    Параметры:
    
    url0 - ссылка на новость (строка).
    """
    page0 = requests.get(url0)
    soup0 = BeautifulSoup(page0.text, 'html')
    
    try:
        author = soup0.find_all('meta', {'name' : 'author'})[0].get('content')
    except IndexError:
        author = None
    date = soup0.find_all('meta', {'itemprop' : 'datePublished'})[0].get('content')
    title = soup0.find_all('meta', {'property' : 'og:title'})[0].get('content')
 
    
    return url0, date, author, title

In [50]:
help(GetNews)

Help on function GetNews in module __main__:

GetNews(url0)
    Возвращает кортеж с url0, date, author, title.
    Параметры:

    url0 - ссылка на новость (строка).



Уфф. Осталось применить ее в цикле. Но давайте не будем спешить: импортируем функцию `sleep` для задержки, чтобы на каждой итерации цикла, прежде чем перейти к следующей новости, Python ждал несколько секунд. Во-первых, это нужно, чтобы сайт «не понял», чтобы мы его грабим, да еще автоматически. Во-вторых, с небольшой задержкой всегда есть гарантия, что страница прогрузится (сейчас это пока не очень важно, но особенно актуально будет, когда будем обсуждать встраивание в браузер с Selenium). Приступим.

In [51]:
from time import sleep

In [52]:
len(urls)

50

In [54]:
from tqdm.notebook import tqdm

In [61]:
news = [] # это будет список из кортежей, в которых будут храниться данные по каждой новости

for link in tqdm(urls[:10], desc='Скачано сайтов'):
    print(link)
    res = GetNews(link)
    news.append(res)
        
    sleep(1) # задержка в 3 секунды

Скачано сайтов:   0%|          | 0/10 [00:00<?, ?it/s]

https://nplus1.ru/news/2026/11/24/ushkanik
https://nplus1.ru/news/2026/11/25/fda-venglustat
https://nplus1.ru/news/2026/11/25/toba-super-eruption
https://nplus1.ru/news/2026/11/25/archaeospirostoma-flinti
https://nplus1.ru/news/2026/11/25/fda-ribitol
https://nplus1.ru/news/2026/11/25/mrdaja
https://nplus1.ru/news/2026/11/24/prehospital-antibiotics-sepsis
https://nplus1.ru/news/2026/11/24/pickleball-traumatic-glaucoma
https://nplus1.ru/news/2026/11/24/islamic-era-shipwreck
https://nplus1.ru/news/2026/11/24/solo-navigation-in-meerkats


In [56]:
news[1]

('https://nplus1.ru/news/2026/11/23/infants-in-himera',
 '2026-11-23',
 'Михаил Подрезов',
 'Малыши из\xa0древней Гимеры попили грудное молоко до\xa0четырехлетнего возраста')

Так теперь выглядит первый элемент списка:

Импортируем `pandas` и создадим датафрейм из списка кортежей: 

In [57]:
import pandas as pd

In [58]:
news

[('https://nplus1.ru/news/2026/11/23/bronze-neck-ring-from-swiss-alps',
  '2026-11-23',
  'Михаил Подрезов',
  'На\xa0альпийском перевале нашли древнюю бронзовую гривну'),
 ('https://nplus1.ru/news/2026/11/23/infants-in-himera',
  '2026-11-23',
  'Михаил Подрезов',
  'Малыши из\xa0древней Гимеры попили грудное молоко до\xa0четырехлетнего возраста')]

In [62]:
df = pd.DataFrame(news)

In [60]:
df

,0,1,2,3
0,https://nplus1.ru/news/2026/11/23/bronze-neck-...,2026-11-23,Михаил Подрезов,На альпийском перевале нашли древнюю бронзовую...
1,https://nplus1.ru/news/2026/11/23/infants-in-h...,2026-11-23,Михаил Подрезов,Малыши из древней Гимеры попили грудное молоко...


In [63]:
df.head(11)

,0,1,2,3
0,https://nplus1.ru/news/2026/11/24/ushkanik,2026-11-24,Михаил Подрезов,Петроглиф носорога назвали изображением бизона
1,https://nplus1.ru/news/2026/11/25/fda-venglustat,2026-11-25,Слава Гоменюк,FDA одобрило первое лекарство для лечения невр...
2,https://nplus1.ru/news/2026/11/25/toba-super-e...,2026-11-25,Михаил Подрезов,Объем суперизвержения Тобы оценили в 3170 куби...
3,https://nplus1.ru/news/2026/11/25/archaeospiro...,2026-11-25,Сергей Коленов,Волосатую улитку из бирманского янтаря назвали...
4,https://nplus1.ru/news/2026/11/25/fda-ribitol,2026-11-25,Слава Гоменюк,FDA одобрило использование пятиатомного спирта...
5,https://nplus1.ru/news/2026/11/25/mrdaja,2026-11-25,Михаил Подрезов,В Македонии нашли бронзовый меч микенского типа
6,https://nplus1.ru/news/2026/11/24/prehospital-...,2026-11-24,Слава Гоменюк,Введение антибиотика цефтриаксона на догоспита...
7,https://nplus1.ru/news/2026/11/24/pickleball-t...,2026-11-24,Слава Гоменюк,Травма от мяча для пиклбола привела к глаукоме...
8,https://nplus1.ru/news/2026/11/24/islamic-era-...,2026-11-24,Михаил Подрезов,У берегов Мальорки нашли следы кораблекрушения...
9,https://nplus1.ru/news/2026/11/24/solo-navigat...,2026-11-24,Катерина Петрова,Социальные сурикаты нашли дорогу домой поодиночке


Переименуем столбцы в базе.

In [64]:
df.columns = ['link', 'date', 'author', 'title']

In [65]:
df.head(2)

,link,date,author,title
0,https://nplus1.ru/news/2026/11/24/ushkanik,2026-11-24,Михаил Подрезов,Петроглиф носорога назвали изображением бизона
1,https://nplus1.ru/news/2026/11/25/fda-venglustat,2026-11-25,Слава Гоменюк,FDA одобрило первое лекарство для лечения невр...


Всё! Сохраняем датафрейм в файл. Для разнообразия сохраним в Excel:

In [66]:
df.to_excel('nplus-news.xlsx', index=False)